# Hardening the headline — min-separation k-NN + a deployable k-NN

Two experiments, 9 runs, ~6 h. Outputs go to `MyDrive/neural_hydro_runs/paper_completion/`
alongside the previous suite. Idempotent: re-run all after a disconnect and it resumes.

**Why these two, and nothing else.**

The paper's headline is now that discarding the river network and averaging each basin's two
nearest gauges beats the true network (+0.081 vs +0.043, paired +0.034, p=1e-24). Two objections
stand between that result and a submission, and these are the experiments that answer them.

| # | Experiment | Runs | The objection it closes |
|---|---|---|---|
| 1 | **Min-separation k-NN** | 6 | 19 of 366 neighbour pairs sit under 10 km (nearest 1.6 km). Those are plausibly nested or near-duplicate gauges, so a reviewer will say the gain is partly the target measuring itself. Re-running with a 10 km and a 15 km floor settles it. **This is the one that can sink the headline.** |
| 2 | **Deployable k-NN** | 3 | The k-NN result uses *observed* neighbour discharge, so it is oracle-to-oracle. This builds the two-stage predicted-Q version, turning the headline into a model someone could actually run. |

Pre-registrations: `preregistration_knn_hardening.md`.

**Runtime → Change runtime type → T4 GPU → Run all.**

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''
OUT_SUBDIR='paper_completion'
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for cand in AUTO:
        if os.path.isdir(cand): DRIVE_CAMELS_PATH=cand; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')
DRIVE_ROOT='/content/drive/MyDrive/neural_hydro_runs'
DRIVE_OUT=os.path.join(DRIVE_ROOT, OUT_SUBDIR)
os.makedirs(f'{DRIVE_OUT}/topology_ablation/component0', exist_ok=True)
SEEDS=[11,13,17]; MIN_KM=[10,15]
print('CAMELS:',DRIVE_CAMELS_PATH); print('OUT   :',DRIVE_OUT)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs, and link in the runs we already have

`runs/` points at `paper_completion/`. Baselines and the existing k-NN / fullspan runs are
symlinked in so deltas resolve and nothing is retrained.

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_OUT, RR)
NEW=f'{DRIVE_OUT}/topology_ablation/component0'; OLD=f'{DRIVE_ROOT}/topology_ablation/component0'
os.makedirs(NEW,exist_ok=True)
want=['L','L_upQ','L_upQpred','L_upQknn2','L_upQknn4']
linked=0; miss=[]
for cond in want:
    for s in SEEDS:
        nm=f'{cond}_component0_seed{s}'
        if os.path.exists(f'{NEW}/{nm}') or os.path.islink(f'{NEW}/{nm}'): continue
        if os.path.isdir(f'{OLD}/{nm}'): os.symlink(f'{OLD}/{nm}',f'{NEW}/{nm}'); linked+=1
        else: miss.append(nm)
for s in SEEDS:  # fullspan evals feed the deployable build
    nm=f'_Lfullspan_eval_seed{s}'
    if not (os.path.exists(f'{NEW}/{nm}') or os.path.islink(f'{NEW}/{nm}')):
        if os.path.isdir(f'{OLD}/{nm}'): os.symlink(f'{OLD}/{nm}',f'{NEW}/{nm}'); linked+=1
        else: miss.append(nm)
print('linked:',linked)
if miss: print('MISSING (may block a delta or the deployable build):',miss)

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:',torch.cuda.get_device_name(0))

## Cell 7 — Helpers

In [ ]:
%cd {REPO_DIR}
import json, pickle, numpy as np, pandas as pd, subprocess, time
from pathlib import Path
from scipy.stats import wilcoxon
FEAT='experiments/topology_ablation/features'
P1=Path('topology_analysis/phase1_network_discovery/outputs')
B=f'{REPO_DIR}/runs/topology_ablation/component0'
N_BASINS=183
def named_ok(p,n_expected=N_BASINS):
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected}'); return False
    return True
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
_f=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
CONN=sorted([b for b,v in _f.items() if float(np.nanmax(np.abs(v.values)))>0]); ALLB=sorted(_f.keys())
def paired(cond,s,ref='L',bas=None):
    A,L=nse(cond,s),nse(ref,s)
    if A is None or L is None: return None
    bs=[b for b in (bas or CONN) if b in A.index and b in L.index]
    return (A[bs]-L[bs]).values
def RUNNER(cond,seed,feature):
    if done(cond,seed): print(f'  [skip] {cond} seed {seed}'); return True
    print(f'  [run ] {cond} seed {seed} ...',flush=True); t0=time.time()
    r=subprocess.run(['python','experiments/topology_ablation/run_upstream_feature.py',
        '--network','component0','--seed',str(seed),'--device','cuda:0','--epochs','30',
        '--feature-file',feature,'--cond-name',cond],capture_output=True,text=True)
    ok=done(cond,seed); print(f'  [{"ok" if ok else "FAIL"}] {cond} seed {seed} ({time.time()-t0:.0f}s)')
    if not ok: print(r.stdout[-1200:]); print(r.stderr[-1200:])
    return ok
RESULTS={}
print('ready | connected',len(CONN))

# Experiment 1 — Min-separation k-NN (6 runs)

Rebuilds the nearest-gauge input with a floor on how close a neighbour may be, at 10 km and 15 km.
If the advantage over the river network survives both floors, the near-duplicate-gauge objection is
closed. If it collapses, the headline was partly leakage and the paper must say so.

In [ ]:
%cd {REPO_DIR}
for m in MIN_KM:
    fp=f'{FEAT}/upstream_q_knn2min{m}km_component0_lag1.p'
    if not named_ok(fp):
        print(f'building min-{m}km feature')
        !python experiments/topology_ablation/build_distance_control.py --network component0 --mode knn --knn-k 2 --knn-min-km {m} --lag-days 1
    print(f'  min-{m}km ok:',named_ok(fp))
for m in MIN_KM:
    fp=f'{FEAT}/upstream_q_knn2min{m}km_component0_lag1.p'
    for s in SEEDS: RUNNER(f'L_upQknn2min{m}km',s,fp)

# Experiment 2 — Deployable k-NN (3 runs)

The k-NN result uses observed neighbour discharge. This builds the two-stage version: stage one
predicts each basin's discharge from its own forcings over the full record, and those *predicted*
series are averaged over the two nearest gauges. No observed discharge is needed at inference.
Uses each seed's own stage-one model, as the network-derived deployable input does.

In [ ]:
%cd {REPO_DIR}
# emit the knn2 edge list the deployable builder consumes
if not os.path.isfile(P1/'component0_edges_knn2.csv'):
    !python experiments/topology_ablation/build_distance_control.py --network component0 --mode knn --knn-k 2 --dry-run
print('knn2 edge list:',os.path.isfile(P1/'component0_edges_knn2.csv'))
for s in SEEDS:
    fp=f'{FEAT}/upstream_q_pred_knn2_component0_seed{s}_lag1.p'
    if not named_ok(fp):
        print(f'building deployable kNN feature, seed {s}')
        !python experiments/topology_ablation/build_predicted_upstream_q.py --network component0 --seed {s} --lag-days 1 --edge-file topology_analysis/phase1_network_discovery/outputs/component0_edges_knn2.csv --tag knn2
    print(f'  seed {s} feature ok:',named_ok(fp))
for s in SEEDS:
    RUNNER('L_upQpredknn2',s,f'{FEAT}/upstream_q_pred_knn2_component0_seed{s}_lag1.p')

# Report

In [ ]:
%cd {REPO_DIR}
line='='*72
def blk(t): print('\n'+line+'\n'+t+'\n'+line)

blk('1. MIN-SEPARATION kNN — is the headline leakage?')
def cs(c_,bas=None):
    per=[np.median(paired(c_,s,bas=bas)) for s in SEEDS if paired(c_,s,bas=bas) is not None]
    return (np.mean(per),per) if per else (float('nan'),[])
g,gp=cs('L_upQ'); k,kp=cs('L_upQknn2')
print(f'  true network      {g:+.4f}   {[f"{x:+.3f}" for x in gp]}')
print(f'  kNN2 (no floor)   {k:+.4f}   {[f"{x:+.3f}" for x in kp]}')
rows={}
for m in MIN_KM:
    v,per=cs(f'L_upQknn2min{m}km')
    rows[m]=v
    print(f'  kNN2 min {m:2d} km    {v:+.4f}   {[f"{x:+.3f}" for x in per]}')
# paired vs the true network
for m in MIN_KM:
    pool=[]
    for s in SEEDS:
        A,G=nse(f'L_upQknn2min{m}km',s),nse('L_upQ',s)
        if A is None or G is None: continue
        bs=[b for b in CONN if b in A.index and b in G.index]
        pool+=list((A[bs]-G[bs]).values)
    if pool:
        print(f'\n  min {m} km vs true network: paired median {np.median(pool):+.4f}, p={wilcoxon(pool)[1]:.2e}, '
              f'kNN better on {np.mean(np.array(pool)>0)*100:.0f}%')
        RESULTS[f'knn_min{m}_vs_graph']=float(np.median(pool))
worst=min(rows.values()) if rows else float('nan')
print('\n  === VERDICT ===')
if rows and worst>g+0.005:
    print(f'  HOLDS. Even with a {max(MIN_KM)} km floor the nearest-gauge input beats the network')
    print('  ({:+.4f} vs {:+.4f}). The near-duplicate-gauge objection is closed.'.format(worst,g))
elif rows and worst<g:
    print('  FALSIFIED. Excluding close gauges drops kNN below the network. The headline was')
    print('  substantially leakage and the paper must be rewritten around that.')
else:
    print('  PARTIAL — advantage shrinks but survives; report the floored numbers as the headline.')

blk('2. DEPLOYABLE kNN — does it work without observed flow?')
r,rp=cs('L_upQpred'); dk,dkp=cs('L_upQpredknn2')
print(f'  network deployable (L_upQpred)   {r:+.4f}   {[f"{x:+.3f}" for x in rp]}')
print(f'  kNN deployable (L_upQpredknn2)   {dk:+.4f}   {[f"{x:+.3f}" for x in dkp]}')
if not np.isnan(dk) and not np.isnan(r):
    pool=[]
    for s in SEEDS:
        A,Rr=nse('L_upQpredknn2',s),nse('L_upQpred',s)
        if A is None or Rr is None: continue
        bs=[b for b in CONN if b in A.index and b in Rr.index]
        pool+=list((A[bs]-Rr[bs]).values)
    if pool:
        print(f'\n  paired kNN-deployable − network-deployable: {np.median(pool):+.4f}, p={wilcoxon(pool)[1]:.2e}')
        RESULTS['knn_deployable_vs_network_deployable']=float(np.median(pool))
    print('\n  === VERDICT ===')
    if dk>r: print('  The nearest-gauge advantage SURVIVES the two-stage step. The headline becomes a')
    else: print('  The advantage does NOT survive prediction. Report kNN as an oracle-only finding and')
    print('  deployable claim, not an oracle-only one.' if dk>r else '  keep the deployable claim on the network-derived input.')

blk('SUMMARY'); print(json.dumps(RESULTS,indent=2,default=float))

## Persistence check

In [ ]:
exp={f'L_upQknn2min{m}km':SEEDS for m in MIN_KM}; exp['L_upQpredknn2']=SEEDS
ok=miss=0
for cond,ss in exp.items():
    for s in ss:
        p=f'{DRIVE_OUT}/topology_ablation/component0/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
        good=os.path.isfile(p); ok+=good; miss+=(not good)
        print(f'  {"OK  " if good else "MISS"} {cond} seed {s}')
print(f'\n{ok} present, {miss} missing (expect 9)')
with open(f'{DRIVE_OUT}/verdicts_hardening.json','w') as f: json.dump(RESULTS,f,indent=2,default=float)
print('wrote',f'{DRIVE_OUT}/verdicts_hardening.json')

## Done

Paste the **report** and the **persistence check** back.

Experiment 1 decides whether the paper's headline survives contact with its sharpest objection.
Experiment 2 decides whether that headline can be stated as a deployable result or only as an
oracle bound.